# Friend A — B35 divisions: A0 audit + Step A daughter recovery

Этот ноутбук соответствует последнему верхнему блоку `PROMPT_FRIEND_A.md`: перед восстановлением дочерей и обучением pair-модели он пересчитывает DivNet-префильтр и потолок детекции официальным оптимальным сопоставлением и окном деления. Старые допущения `top 3%` и `121/151` не используются как готовые параметры.

A0 работает на CPU. После принятого ответа на A0 ноутбук выполняет Step A: один full-frame проход public primary при threshold 0.90 и NMS 3 мкм, кэширует пики и проверяет новый gate 144/151 при не более 0.3% кандидатов на видео. Оба этапа возобновляются по видео. Никакого обучения, отправки на leaderboard или загрузки датасета этот ноутбук не выполняет.


In [ ]:
from pathlib import Path
import importlib
import sys

start = Path.cwd().resolve()
roots = [start, start / '.github-push-worktree']
for parent in start.parents:
    roots.extend([parent, parent / '.github-push-worktree'])
REPO_ROOT = next((p for p in roots if (p / 'TEAM.md').is_file() and (p / 'tools' / 'audit_b35.py').is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError('Repository root not found. Start Jupyter from C:\\Projects\\CellTracker.')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from experiments.local_friend_a_b32_divpair import b35_runner
importlib.reload(b35_runner)
from experiments.local_friend_a_b32_divpair.b35_runner import (
    B35Config, build_friend_c_package, clear_stop, compact_status,
    clear_step_a_stop, compact_step_a_status, preflight, preflight_step_a,
    request_step_a_stop, request_stop, run_a0, run_smoke,
    run_step_a, run_step_a_smoke,
)
print('Repository:', REPO_ROOT)


In [ ]:
cfg = B35Config.from_environment(REPO_ROOT)
cfg.serialisable()


In [ ]:
# Run All deliberately clears only an old stop request; completed video reports remain.
clear_stop(cfg)
check = preflight(cfg)
check


Smoke использует два видео разных эмбрионов с шестью размеченными делениями. Его числа не являются результатом B35.


In [ ]:
smoke_state = run_smoke(cfg)
smoke_state


Следующая ячейка выполняет A0 на всех 199 видео. После каждого видео сохраняются JSON и сводка. При повторном запуске готовые видео проверяются по fingerprint и пропускаются. Полный отчёт появится также в `experiments/local_friend_a_b32_divpair/results/b35_audit.{json,md}`.


In [ ]:
a0_state = run_a0(cfg)
compact_status(cfg)


In [ ]:
report_path = cfg.audit_dir / 'b35_audit.md'
if report_path.is_file():
    print(report_path.read_text(encoding='utf-8'))
compact_status(cfg)


## Step A — full-frame primary 0.90

A0 уже принят: baseline оконного охвата 136/151. Этот этап использует только public primary, тот же публичный detection TTA и NMS 3 мкм. Пики с threshold 0.90 сохраняются по видео; готовые видео при повторном запуске пропускаются. Gate полного запуска: минимум 144/151 и не более 0.3% новых кандидатов в каждом видео. Независимо от gate следующим будет Step B; Step C отменён.


In [ ]:
clear_step_a_stop(cfg)
step_a_check = preflight_step_a(cfg)
step_a_check


GPU smoke обрабатывает два уже выбранных видео разных эмбрионов. Он пишет те же кэши, поэтому полный запуск повторно их не считает.


In [ ]:
step_a_smoke = run_step_a_smoke(cfg)
step_a_smoke


Следующая ячейка — долгий GPU-прогон всех 199 видео. Останавливать безопасно через Interrupt Kernel: законченные видео остаются в кэше. Повторный запуск этой же ячейки продолжает расчёт.


In [ ]:
step_a_state = run_step_a(cfg)
compact_step_a_status(cfg)


In [ ]:
step_a_report = cfg.step_a_dir / 'summary.md'
if step_a_report.is_file():
    print(step_a_report.read_text(encoding='utf-8'))
compact_step_a_status(cfg)


Пакет для Friend C собирается отдельно после A0. Он содержит `raw/`, `ckpt/`, `dc/`, `divnet/` и GT, но не содержит изображения или веса. Выставь флаг в `True`, когда нужен ZIP; автоматической публикации на Kaggle нет.


In [ ]:
BUILD_FRIEND_C_PACKAGE = False
if BUILD_FRIEND_C_PACKAGE:
    print('Package:', build_friend_c_package(cfg))


In [ ]:
# Выполняй отдельно только для мягкой остановки между видео.
REQUEST_STOP = False
if REQUEST_STOP:
    print('Stop requested:', request_stop(cfg))


In [ ]:
# Выполняй отдельно только для мягкой остановки Step A между видео.
REQUEST_STEP_A_STOP = False
if REQUEST_STEP_A_STOP:
    print('Step A stop requested:', request_step_a_stop(cfg))
